# 06. 추론과 행동 배열 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

추론은 행동 예측까지입니다. infer 모드에서는 로봇에 행동을 적용하지 않습니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab06_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![추론과 행동 배열의 입력·처리·출력](../교재그림/guided-vla/day5-06.svg)

load_policy는 가중치와 전후처리기를 준비합니다. torch.inference_mode에서 예측한 뒤 postprocessor로 행동을 복원하고 shape와 유한성을 검사합니다. 저장 배열의 첫 축은 묶음 수입니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 모델 적재와 추론은 다른 단계

- 가중치 로드: 디스크의 학습 결과를 정책 객체로 준비
- eval(): 평가 동작 설정; inference_mode(): 기울기 기록 생략
- 전처리·정책·후처리를 한 경로로 사용
- infer는 예측 배열만 저장하며 로봇을 움직이지 않음

모델 적재는 설정과 가중치를 읽고 연산에 사용할 정책을 구성하는 준비 과정입니다. 추론은 이미 준비된 정책에 현재 입력을 넣어 출력을 계산하는 단계입니다. 첫 요청이 느리다는 현상을 설명할 때 두 시간을 구분해야 합니다. load_policy의 strict=True는 가중치와 모델 구조의 불일치를 조용히 넘기지 않도록 합니다.

config.load_vlm_weights=False는 완성 정책에 들어 있는 backbone 가중치를 중복 다운로드하지 않는 설정입니다. 영상·언어 부분을 쓰지 않겠다는 뜻이 아닙니다. policy.eval()과 torch.inference_mode()도 같은 기능이 아닙니다. 전자는 평가 시 동작을, 후자는 기울기 추적 여부를 제어합니다.

이 수업에서는 추론 중 가중치를 갱신하지 않습니다. 같은 체크포인트의 출력을 읽고 실행 조건을 비교합니다. 예측값의 분포를 본다는 것과 새로운 정책을 학습한다는 것은 다릅니다.


### 2. 실행 아키텍처: GPU 예측과 CPU 저장

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-06.svg)

- check_model → load_policy → pre/post 준비
- pre(make_batch(obs,task)) → predict_action_chunk
- post → detach → CPU → NumPy → batch 첫 항목
- 50×7 검사 → action-chunks.npy·result.json

infer는 inspect와 같은 초기 관측을 만든 다음 정책을 로드하고 policy.reset()을 호출합니다. 정책 내부 상태를 초기화하는 reset과 물리 환경의 env.reset은 서로 다릅니다. 그 뒤 pre(make_batch(...))로 입력을 준비하고 predict_action_chunk를 실행합니다.

출력은 postprocessor를 거친 뒤 detach().cpu().numpy()[0]으로 옮깁니다. detach는 계산 그래프에서 분리, cpu는 저장·분석 가능한 메모리로 이동, numpy는 배열 표현 변환, [0]은 배치 축의 첫 항목 선택입니다. 이 네 동작을 단순히 숫자 정리라고 뭉뚱그리지 않고 축과 메모리 경계를 구분합니다.

검사할 chunk는 (50,7)이고 모든 값이 유한해야 합니다. 통과하면 chunks 목록에 넣고 infer 모드이므로 행동 적용 for문에 들어가기 전에 종료합니다. 로봇이 움직이지 않고 action-chunks.npy가 생성되는 것이 이 모드의 정상 결과입니다.


### 3. 코드 추적: 배열과 시간 측정의 범위

- 저장 shape (1,50,7): 묶음1·미래50·행동7
- 첫3행은 초기 예측, min/max는 묶음 내 성분 범위
- chunk_seconds는 정책·후처리·CPU 이동 등을 포함
- make_batch/pre와 모델 로드 시간은 이 구간 밖

run_episode는 pre(make_batch(...)) 뒤 CUDA synchronize를 호출하고 tick을 기록합니다. 그 다음 정책 예측, post, detach·CPU 이동·NumPy 변환과 동기화를 마친 시각 차이를 chunk_seconds에 추가합니다. 따라서 순수 신경망 kernel 시간만 측정한 값이 아닙니다. 코드에서 타이머 시작과 끝을 읽어 포함 범위를 설명합니다.

model_load_seconds는 별도이며 execution_seconds도 infer/rollout 실행 구간 전체를 보는 다른 값입니다. 재사용된 완료 결과에서 같은 숫자를 다시 출력하는 것은 새 측정이 아닙니다. GPU 계산은 비동기일 수 있어 synchronize가 측정 경계를 맞추는 데 사용된다는 점도 함께 짚습니다.

행동 그래프는 열마다 다른 의미를 가집니다. 7열을 모두 같은 물리 단위의 값처럼 평균내지 않습니다. 큰 변화가 나왔다는 사실만으로 실패를 단정할 수 없고, 적용 후 관측과 성공 조건을 확인해야 합니다. infer에서는 그 적용 단계가 아직 없습니다.


### 4. 출력 오류와 정책 변경 시 주의점

- 형상 오류·NaN은 행동 적용 전에 거부됩니다.
- post를 생략하면 모델 내부 출력 표현이 남을 수 있습니다.
- 저장된 예측과 env.step의 clip 이후 값은 다를 수 있습니다.
- 체크포인트 교체는 processor·통계·계약까지 함께 검토

shape가 (50,7)이 아니거나 NaN·Inf가 있으면 runner는 ValueError로 중단하고 incomplete 기록을 남깁니다. 숫자가 보인다는 이유로 그대로 환경에 전달하지 않습니다. min/max 표는 이 값을 읽기 쉽게 보여 주지만 유한성 검사와 의미 계약을 대체하지 않습니다.

저장 배열은 postprocessor 이후의 예측값입니다. 실제 rollout에서는 각 action을 np.clip(action,-1,1)한 뒤 env.step에 전달하므로 범위를 넘는 예측은 적용값과 다릅니다. 그래프에 나타난 값과 실제 환경 입력을 동일시할 때 이 경계를 확인합니다.

직접 다른 체크포인트를 사용하려면 가중치뿐 아니라 config, 전후처리기, 상태 통계, 카메라 매핑, chunk 형상을 함께 확인해야 합니다. strict를 끄거나 shape 검사만 없애서 실행되게 만드는 것은 호환성을 증명하지 않습니다. 수업에서는 준비된 정책의 입력과 출력 경계를 충분히 읽는 데 집중합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/run_episode.py : predict_action_chunk](scripts/run_episode.py) |
| 바꾸어 볼 조건 | `같은 결과의 앞3행·성분별 min/max 표시` |
| 예상되는 차이 | 7개 행동 성분의 범위를 자동 표로 읽고 이동·회전·그리퍼 그래프를 봅니다. 큰 변화가 곧 작업 실패라는 뜻은 아닙니다. |
| 직접 구현·수정할 때 | 모델 로드와 chunk_seconds를 섞지 않습니다. 저장된 완료 결과 재사용은 새 측정이 아닙니다. 시뮬레이터는 추론 중 대기합니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


### 실제 코드 읽기 — 입력·처리·출력 연결

load_policy는 가중치와 전후처리기를 준비합니다. torch.inference_mode에서 예측한 뒤 postprocessor로 행동을 복원하고 shape와 유한성을 검사합니다. 저장 배열의 첫 축은 묶음 수입니다.


In [ ]:
from pathlib import Path
from IPython.display import Code, display
import ast
source_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'10.VLA_v2/scripts').is_dir())
source_path = source_root/'10.VLA_v2'/'scripts/course_runtime.py'
text = source_path.read_text()
# 함수 위치를 AST로 찾아 해당 함수 전체를 표시합니다. 파일을 import하거나 모델을 로드하지 않습니다.
functions = [node for node in ast.parse(text).body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))]
for node in functions:
    if node.name in ['load_policy']:
        print(source_path, '함수:', node.name, '줄:', node.lineno)
        display(Code('\n'.join(text.splitlines()[node.lineno-1:node.end_lineno]), language='python'))


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [5일차 실행 안내](README.md)를 참고합니다.

**5일차 환경:** `nepes-course:physical-ai-vla-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 같은 날에는 준비된 개인 VLA 환경을 유지합니다.

**실행 전 확인 — 컨테이너 내부 Terminal:**

```bash
cd /workspace
/opt/course/bin/python 10.VLA_v2/scripts/preflight.py
```

Python 커널도 `/opt/course/bin/python`을 사용합니다. 강사 배포 `Data/models/smolvla-libero-probe`(LIBERO용 정책 revision `31d453f7edd78c839a8bbc39744a292686daf0de`), `Data/cache/smolvla-probe/huggingface`, `Data/cache/smolvla-probe/libero`와 LIBERO 자산이 필요합니다. 파일 누락은 호스트 개인 Data의 마운트·배포본을 확인합니다. 수강생이 `pip install`, Docker build, 모델 다운로드를 하지 않습니다. CUDA/EGL 문제는 전용 이미지·NVIDIA runtime과 다른 GPU 작업을 확인합니다.

**실행과 저장:** 04부터 08까지의 노트북의 `SESSION = 'v2-session-01'`은 같은 회차이며 결과는 `Data/logs/day5/v2-session-01/<실행이름>/`에 남습니다. 새 비교 전체는 모든 후속 노트북의 SESSION을 함께 `session-02` 등으로 바꿉니다. `run_lesson`은 동일 조건의 `completed` 결과를 읽고 없을 때 실행합니다. 미완료·조건 불일치는 기존 파일을 삭제하거나 성공값으로 바꾸지 말고 새 회차/실행 이름을 사용합니다. JSON은 있는데 필요한 PNG/NPZ/NPY/영상이 없으면 부분 손상으로 보고 새 회차에서 다시 생성합니다.

**한 번에 한 정책 프로세스만 실행합니다.** 이전 실행이 끝난 뒤 다음 셀을 실행하고 다른 노트북의 infer/rollout을 동시에 시작하지 않습니다. 실행 중단 후에는 해당 Terminal이나 실행 로그에서 자식 프로세스까지 종료됐는지 확인합니다. `status=completed`는 절차 완료, rollout의 `success=true`는 환경 과제 성공이며 inspect/infer의 `success=false`는 아직 작업을 하지 않았다는 뜻입니다.


## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

<!-- procedure:novnc-1744bd:start -->
![noVNC의 정상 대기와 완료](../교재그림/절차/novnc.svg)

03의 준비 검사와 05의 저장 배열 읽기는 새 실험 창을 열지 않습니다. 기존 완료 결과 재사용에도 새 창이 없습니다.
<!-- procedure:novnc-1744bd:end -->


JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

**이 노트북에서는 초기 장면과 모델 로드·추론 상태를 봅니다.** 행동을 예측·저장하지만 환경에 적용하지 않으므로 로봇은 움직이지 않습니다. 완료 창을 닫고 행동 배열·그래프를 확인합니다.

설정 셀의 `GUI`는 자동으로 켜집니다. 창 없이 시간 비교를 할 때는 `GUI = False`로 바꾸고 설정 셀을 다시 실행합니다. CLI 대체 실행은 `run_episode.py`에 `--gui`를 붙입니다. 노트북과 CLI를 동시에 실행하지 않습니다.

환경 초기화가 끝난 뒤 실제 초기 장면에서 창이 열립니다. 초기화 중 오류는 노트북 로그에서 확인합니다. 화면에는 실행 이름·모드·현재 상태·스텝·추론 횟수·적용 길이·직전 추론 초가 표시됩니다.

| 화면 상태 | 의미 |
|---|---|
| `INITIAL OBSERVATION` | 초기 관측 준비 완료 |
| `MODEL LOADING` | 모델 로드 중, 환경은 대기 |
| `INFERENCE` | 행동 계산 중, 환경은 대기 |
| `ACTION EXECUTION` | 예측 명령을 환경에 적용 중 |
| `FINISHED` | 종료 이유와 성공 여부 표시. 마지막 장면이며 더 움직이지 않음 |
| `INCOMPLETE / ERROR` | 미완료. 노트북의 원인 로그 확인 |

**정상 종료 화면을 관찰한 뒤 창을 닫으면 셀이 끝납니다.** 결과와 영상은 종료 화면 대기 전에 저장됩니다. 실행 도중 창을 닫으면 다음 안전한 처리 지점에서 사용자 중단으로 기록합니다. GPU 추론·모델 로딩 중에는 그 작업이 반환될 때까지 중단 반영이 늦을 수 있습니다. 창 조작은 표시용 복사본에만 적용되어 정책 카메라와 실제 실험 상태를 바꾸지 않습니다.

**기존 완료 결과를 읽으면 새 GUI는 열리지 않습니다.** 새 시연은 사용하지 않은 실행 이름으로 수행하고 변수·파일 경로도 함께 맞춥니다. GUI 사용 여부는 `result.json`의 `gui_enabled`에 남습니다. 화면 표시 비용이 있으므로 GUI 실행과 비GUI 실행의 시간을 같은 조건처럼 비교하지 않습니다.


**브라우저 탭을 닫아도 실험은 종료되지 않습니다.** 다시 같은 주소로 접속하면 진행 중 화면으로 돌아옵니다. FINISHED 후 화면 안의 MuJoCo 창을 닫아야 셀이 끝납니다. noVNC 연결과 탭은 다음 실험까지 유지하세요. 다른 학생 화면이 보이면 포트를 확인합니다.


## 이전 자료의 위치와 필요한 입력 계약

[05_observation_action.ipynb의 「개념」](05_observation_action.ipynb)에 영상2·상태8·행동7이 있습니다. 다시 쓰면 raw RGB 영상은 각각 `(256,256,3)` uint8이며 방향 반전·CHW float 변환과 processor 정규화를 거칩니다. 상태는 위치3+axis-angle3+그리퍼 관절2, 행동은 OSC 끝단 이동3+회전3+그리퍼1이며 절대 좌표가 아닙니다.

이 노트북의 `run_lesson('infer','infer-01')`는 자체 환경 reset으로 초기 관측과 지시를 만들고 정책을 로드하므로 inspect-01 파일이나 05의 변수는 필요하지 않습니다. 필수 체크포인트·캐시는 시작 절에 있습니다. 출력 `Data/logs/day5/<SESSION>/infer-01/action-chunks.npy`는 `(1,50,7)`의 postprocessor 적용 후 행동 배열, result.json의 chunk_seconds는 각 추론 소요 초 목록입니다. 새 측정은 `infer-02` 같은 새 이름을 쓰며 기존 완료 결과를 읽는 것은 재측정이 아닙니다.


## 개념

정책을 strict=True로 로드합니다. load_vlm_weights=False는 전체 정책에 이미 포함된 backbone 가중치를 다시 다운로드하지 않는 설정이며 가중치를 생략하는 것이 아닙니다.

아래는 실제 환경 관측으로 한 번 추론하고 행동을 저장합니다. 아직 로봇 작업을 수행하지 않습니다. 모델 로드 시간과 행동 묶음 추론 시간을 나누어 읽으세요.

### 단계별 활동

진행: 모델 로드15분 → 실제 입력 추론15분 → 행동 묶음 시각화15분 → 지연·실패 해석15분. 출력의 각 열을 시간축에 대해 그려 변화를 확인합니다. 첫 묶음의 미래 행동은 다음 관측으로 다시 예측되므로 확정된 궤적이 아닙니다. infer-02로 재실행하고 초기화 시간과 추론 시간을 분리해 비교하세요.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
from pathlib import Path
import sys, subprocess, json, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '10.VLA_v2').is_dir())
SCRIPTS = ROOT / '10.VLA_v2/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, lesson_runs, run_or_read_lesson
# 04~08 모두 같은 회차를 사용합니다. 전체 비교를 새로 할 때 session-02 등으로 함께 변경하세요.
GUI = os.environ.get('COURSE_VLA_GUI', '0') == '1'  # 개인 noVNC 화면 사용
SESSION = 'v2-session-01'
RUNS = lesson_runs(DATA, SESSION)
RUNS.mkdir(parents=True, exist_ok=True)
def run_lesson(mode, name, seed=7, execute=10, steps=280, initial_state=0):
    return run_or_read_lesson(SCRIPTS, RUNS, mode, name, seed, execute, steps, initial_state, gui=GUI)


## 예제

<!-- procedure:lesson-42-e862f8:start -->
![예측하고 창을 닫은 뒤 읽기](../교재그림/절차/lesson-42.svg)

기존 완료 결과를 재사용하면 새 창이 없습니다. 새 실행에서는 FINISHED 후 MuJoCo 창을 닫습니다.
<!-- procedure:lesson-42-e862f8:end -->


### 실행 전 확인 — FINISHED 후 MuJoCo 창 닫기

**noVNC에서 `FINISHED`를 확인하면 화면 안의 MuJoCo 창 오른쪽 위 ×를 눌러 닫아주세요.** 창을 닫아야 아래 셀이 완료되고 다음 단계로 진행됩니다. 결과는 이미 저장되었지만, 마지막 장면을 관찰하도록 창을 유지하므로 노트북의 `[*]`와 경과 시간은 계속 표시됩니다. **브라우저 탭만 닫으면 셀은 끝나지 않습니다.** 실행 중 창을 닫으면 중단으로 처리되므로 `FINISHED`를 확인한 뒤 닫습니다.

06에서는 행동을 예측·저장만 하므로 로봇이 움직이지 않는 것이 정상입니다. 완료 창을 닫으면 행동 배열·추론 시간이 출력되고 다음 셀에서 그래프를 확인할 수 있습니다.

기존 완료 결과를 재사용하거나 `GUI=False`로 실행하면 새 GUI 창이 열리지 않으며, 창 닫기를 기다리지 않습니다.


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
# infer는 행동을 예측·저장하지만 env.step을 호출하지 않습니다. 기존 결과 재사용은 재측정이 아닙니다.
result = run_lesson('infer', 'infer-01')
import numpy as np
# 저장 배열을 읽는 단계입니다. 이 호출 자체가 모델 추론이나 새 환경 실행을 시작하지 않습니다.
actions = np.load(RUNS/'infer-01/action-chunks.npy')
print('묶음수, 시점수, 행동차원:', actions.shape)
assert actions.shape == (1,50,7)
assert np.isfinite(actions).all()
print('앞3시점:', actions[0,:3])
print('추론 초:', result['chunk_seconds'])


### 입력과 실행 흐름 확인


In [ ]:
# 제공된 실행 셀입니다. 출력과 파일을 확인하며 위에서 아래로 실행합니다.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2,1,figsize=(10,5))
axes[0].plot(actions[0,:,:3]); axes[0].set_title('Relative translation action (normalized)')
axes[0].legend(['translation x', 'translation y', 'translation z'])
axes[1].plot(actions[0,:,3:]); axes[1].set_title('Rotation and gripper action')
axes[1].legend(['rotation x', 'rotation y', 'rotation z', 'gripper'])
for ax in axes:
    ax.set_xlabel('Future action index (0–49)')
    ax.set_ylabel('Postprocessed action value')
    ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()


### 코드 해설 — 입력부터 결과까지

**행렬 읽기:** actions shape `(1,50,7)`에서 1은 저장 묶음 수, 50은 미래 행동 시점, 7은 행동 성분입니다. `actions[0].min(axis=0)`은 50개 시점을 줄여 각 성분의 최솟값 7개를 줍니다. axis=1이면 시점마다 성분을 섞어 다른 질문에 답합니다.

한 관측에서 50개를 예측했으므로 카메라를 50번 다시 본 것이 아닙니다. 모델 로드 시간과 chunk_seconds를 구분하고, 유한한 행렬이 나왔다는 것과 물체 조작 성공을 구분합니다.

**확인 질문:** 그리퍼 열이 바뀌는 시점을 보았다고 실제 물체를 잡았음을 확정할 수 있나요?

**적용·재확인:** 자동 출력과 실제 파일의 조건을 함께 확인합니다. 완료된 동일 조건 결과는 다시 읽고, 새 실험은 새 실행 이름을 사용합니다. `SESSION`을 바꾸면 후속 노트북도 맞춥니다. 형상 검사·실행 완료·작업 성공을 따로 확인하며, 예시 수치를 자기 측정값으로 복사하지 않습니다.


## 실행 실습 — 같은 결과의 앞3행·성분별 min/max 표시

7개 행동 성분의 범위를 자동 표로 읽고 이동·회전·그리퍼 그래프를 봅니다. 큰 변화가 곧 작업 실패라는 뜻은 아닙니다.


In [ ]:
from IPython.display import Markdown, display
# 행동 성분별 통계를 자동 계산합니다. 빈 notes를 채우거나 수치를 옮겨 적지 않습니다.
labels = ['translation_x', 'translation_y', 'translation_z', 'rotation_x', 'rotation_y', 'rotation_z', 'gripper']
chunk = actions[0]
lines = ['성분 | 첫 값 | 최솟값 | 최댓값', '--- | --- | --- | ---']
for index, label in enumerate(labels):
    lines.append(f'{label} | {chunk[0,index]:.4f} | {chunk[:,index].min():.4f} | {chunk[:,index].max():.4f}')
display(Markdown('\n'.join(lines)))
print('환경 준비 초:', result.get('environment_setup_seconds', '미기록'))
print('모델 로드 초:', result.get('model_load_seconds', '미기록'))
print('행동 묶음 추론 초:', result['chunk_seconds'])
print('infer는 행동을 적용하지 않습니다:', result['termination_reason'])


## 결과 해설과 다음 단계

추론은 행동 예측까지입니다. infer 모드에서는 로봇에 행동을 적용하지 않습니다.

모델 로드와 chunk_seconds를 섞지 않습니다. 저장된 완료 결과 재사용은 새 측정이 아닙니다. 시뮬레이터는 추론 중 대기합니다.

다음에는 예측한 50개 중 앞부분을 실제 적용하고 다시 관측합니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
